In [ ]:
!pip uninstall -y torchvision
!pip install torchvision==0.20.1+cu130 --index-url https://download.pytorch.org/whl/cu130
!pip install -q evaluate

Looking in indexes: https://download.pytorch.org/whl/cu130
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.1+cu130 (from versions: 0.1.6, 0.2.0, 0.24.0+cu130, 0.24.1+cu130, 0.25.0+cu130, 0.26.0+cu130)
ERROR: No matching distribution found for torchvision==0.20.1+cu130


In [ ]:
# =========================================
# 1️⃣ Install Libraries
# =========================================
!pip install -U transformers datasets seqeval torch tqdm

# =========================================
# 2️⃣ Imports
# =========================================
import json
from tqdm import tqdm
import torch
import evaluate
seqeval_metric = evaluate.load("seqeval")
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
from sklearn.model_selection import train_test_split

# =========================================
# 3️⃣ Load JSONL dataset
# =========================================
dataset_path = "/content/CN-cleaned_file.jsonl"
dataset = []
with open(dataset_path, "r") as f:
    for line in f:
        dataset.append(json.loads(line))

texts = [d["input"] for d in dataset]
labels = [json.loads(d["output"]) for d in dataset]  # [{"entity":..., "label":...}]

# =========================================
# 4️⃣ Label set
# =========================================
label_list = ["TOPIC", "CONCEPT", "METHOD", "ALGORITHM", "OTHER"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}

# Convert entity spans to token-level labels
def encode_labels(text, entities):
    tokens = text.split()
    token_labels = ["O"] * len(tokens)
    for ent in entities:
        ent_tokens = ent["entity"].split()
        ent_label = ent["label"] if ent["label"] in label_list else "OTHER"
        for i in range(len(tokens) - len(ent_tokens) + 1):
            if tokens[i:i+len(ent_tokens)] == ent_tokens:
                token_labels[i:i+len(ent_tokens)] = [ent_label] * len(ent_tokens)
    return token_labels

token_labels = [encode_labels(t, e) for t, e in zip(texts, labels)]

# =========================================
# 5️⃣ Train/Val/Test split
# =========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, token_labels, test_size=0.15, random_state=42
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.33, random_state=42
)

# HuggingFace Dataset
train_dataset = Dataset.from_dict({"tokens": [t.split() for t in train_texts], "ner_tags": train_labels})
val_dataset = Dataset.from_dict({"tokens": [t.split() for t in val_texts], "ner_tags": val_labels})
test_dataset = Dataset.from_dict({"tokens": [t.split() for t in test_texts], "ner_tags": test_labels})

# =========================================
# 6️⃣ Models and Tokenizers
# =========================================
scibert_model_name = "allenai/scibert_scivocab_uncased"
roberta_model_name = "roberta-base"

scibert_tokenizer = AutoTokenizer.from_pretrained(scibert_model_name)
scibert_model = AutoModelForTokenClassification.from_pretrained(
    scibert_model_name, num_labels=len(label_list)
)

roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForTokenClassification.from_pretrained(
    roberta_model_name, num_labels=len(label_list)
)

# ------------------ Set proper label mapping ------------------
scibert_model.config.id2label = id2label
scibert_model.config.label2id = label2id

roberta_model.config.id2label = id2label
roberta_model.config.label2id = label2id

scibert_pipeline = pipeline(
    "ner",
    model=scibert_model,
    tokenizer=scibert_tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

roberta_pipeline = pipeline(
    "ner",
    model=roberta_model,
    tokenizer=roberta_tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

# =========================================
# 7️⃣ Tokenization + label alignment
# =========================================
def tokenize_and_align_labels(examples, tokenizer):
    # Use is_split_into_words=True since input is list of tokens
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        for word_id in word_ids:
            # Assign -100 to special tokens or tokens outside original words
            if word_id is None:
                label_ids.append(-100)
            else:
                # Safety check
                if word_id >= len(labels):
                    label_ids.append(-100)
                else:
                    label_ids.append(label2id.get(labels[word_id], label2id["OTHER"]))
        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

train_dataset_sci = train_dataset.map(
    lambda x: tokenize_and_align_labels(x, scibert_tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names   # <--- remove old columns
)
val_dataset_sci = val_dataset.map(
    lambda x: tokenize_and_align_labels(x, scibert_tokenizer),
    batched=True,
    remove_columns=val_dataset.column_names
)
test_dataset_sci = test_dataset.map(
    lambda x: tokenize_and_align_labels(x, scibert_tokenizer),
    batched=True,
    remove_columns=test_dataset.column_names
)

train_dataset_roberta = train_dataset.map(
    lambda x: tokenize_and_align_labels(x, roberta_tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names
)
val_dataset_roberta = val_dataset.map(
    lambda x: tokenize_and_align_labels(x, roberta_tokenizer),
    batched=True,
    remove_columns=val_dataset.column_names
)
test_dataset_roberta = test_dataset.map(
    lambda x: tokenize_and_align_labels(x, roberta_tokenizer),
    batched=True,
    remove_columns=test_dataset.column_names
)

for d in [train_dataset_sci, val_dataset_sci, test_dataset_sci,
          train_dataset_roberta, val_dataset_roberta, test_dataset_roberta]:
    d.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# =========================================
# 8️⃣ Training Arguments
# =========================================
training_args = TrainingArguments(
    output_dir="./ner_models",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    eval_strategy="epoch",  # ✅ new name
    save_strategy="epoch",
    logging_dir="./logs",
)

# =========================================
# 9️⃣ Metrics
# =========================================
def compute_metrics(p):
    predictions, labels = p
    preds = predictions.argmax(-1)
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    pred_labels = [[id2label[p] for (p, l) in zip(pred, label) if l != -100] for pred, label in zip(preds, labels)]
    return seqeval_metric.compute(predictions=pred_labels, references=true_labels)

# =========================================
# 🔟 Train SciBERT
# =========================================
trainer_sci = Trainer(
    model=scibert_model,
    args=training_args,
    train_dataset=train_dataset_sci,
    eval_dataset=val_dataset_sci,
    compute_metrics=compute_metrics
)
trainer_sci.train()

# =========================================
# 1️⃣1️⃣ Train RoBERTa
# =========================================
trainer_roberta = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=train_dataset_roberta,
    eval_dataset=val_dataset_roberta,
    compute_metrics=compute_metrics
)
trainer_roberta.train()

# =========================================
# 1️⃣2️⃣ Pipelines for Ensemble
# =========================================
scibert_pipeline = pipeline("ner", model=scibert_model, tokenizer=scibert_tokenizer,
                            aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)
roberta_pipeline = pipeline("ner", model=roberta_model, tokenizer=roberta_tokenizer,
                            aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)


# Combine entities with highest confidence, avoid overly long or multi-sentence entities
def combine_entities(entities1, entities2, min_confidence=0.6, max_tokens=15):
    combined = {}
    for e in entities1 + entities2:
        key = e["word"].strip()

        # Skip low-confidence predictions
        if e["score"] < min_confidence:
            continue

        # Skip overly long entities
        if len(key.split()) > max_tokens:
            continue

        # Skip multi-sentence entities (often pipeline merges too much)
        if '.' in key or '\n' in key:
            continue

        # Keep the highest-scoring prediction for each entity
        if key not in combined or combined[key]["score"] < e["score"]:
            combined[key] = {"label": e["entity_group"], "score": e["score"]}

    # Convert to list with rounded scores
    return [{"entity": k, "label": v["label"], "score": round(v["score"], 3)}
            for k, v in combined.items()]

# =========================================
# 1️⃣3️⃣ Ensemble predictions
# =========================================
ensemble_results = []
for item in tqdm(test_texts):
    entities_sci = scibert_pipeline(item)
    entities_roberta = roberta_pipeline(item)
    combined = combine_entities(entities_sci, entities_roberta)
    ensemble_results.append({
        "text": item,
        "entities": combined
    })

# =========================================
# 1️⃣4️⃣ Save results
# =========================================
import json
import numpy as np

def convert_floats(obj):
    if isinstance(obj, dict):
        return {k: convert_floats(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_floats(i) for i in obj]
    elif isinstance(obj, np.float32):
        return float(obj)
    else:
        return obj

# Prepare ensemble results with float conversion
ensemble_results_clean = convert_floats(ensemble_results)

with open("ensemble_entities.json", "w") as f:
    json.dump(ensemble_results_clean, f, indent=4)

print("✅ Ensemble entities saved to ensemble_entities.json")

# =========================================
# 1️⃣5️⃣ Ensemble-level evaluation
# =========================================
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def align_for_seqeval(pred_entities, tokens):
    """
    Converts predicted entities into token-level labels compatible with seqeval.
    O = outside, label = entity label
    """
    token_labels = ["O"] * len(tokens)
    token_texts = tokens  # tokenized as in training (split by whitespace)
    for ent in pred_entities:
        ent_tokens = ent["entity"].split()
        ent_label = ent["label"]
        for i in range(len(token_texts) - len(ent_tokens) + 1):
            if token_texts[i:i+len(ent_tokens)] == ent_tokens:
                token_labels[i:i+len(ent_tokens)] = [ent_label] * len(ent_tokens)
    return token_labels

true_labels_seq = []
pred_labels_seq = []

for text, true_entity_list in zip(test_texts, test_labels):
    tokens = text.split()
    # true labels
    true_labels_seq.append(true_entity_list)

    # predicted labels from ensemble
    idx = test_texts.index(text)
    pred_entities = ensemble_results[idx]["entities"]
    pred_labels_seq.append(align_for_seqeval(pred_entities, tokens))

# Compute metrics
ensemble_precision = precision_score(true_labels_seq, pred_labels_seq)
ensemble_recall = recall_score(true_labels_seq, pred_labels_seq)
ensemble_f1 = f1_score(true_labels_seq, pred_labels_seq)

print("\n📊 Ensemble-level Evaluation on Test Set:")
print(f"Precision: {ensemble_precision:.4f}")
print(f"Recall   : {ensemble_recall:.4f}")
print(f"F1-score : {ensemble_f1:.4f}")

# Optional: full classification report
print("\nClassification Report:")
print(classification_report(true_labels_seq, pred_labels_seq))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored 

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: OTHER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TOPIC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CONCEPT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: METHOD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 76/76 [00:02<00:00, 31.92it/s]

✅ Ensemble entities saved to ensemble_entities.json

📊 Ensemble-level Evaluation on Test Set:
Precision: 0.8650
Recall   : 0.7090
F1-score : 0.7793

Classification Report:
              precision    recall  f1-score   support

       ETHOD       1.00      0.67      0.80         6
      ONCEPT       0.89      0.68      0.77       175
        OPIC       0.81      0.79      0.80        63

   micro avg       0.86      0.71      0.78       244
   macro avg       0.90      0.71      0.79       244
weighted avg       0.87      0.71      0.78       244




/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CONCEPT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TOPIC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: OTHER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: METHOD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


In [2]:
!rm -rf /content/roberta-relation

In [3]:
# =========================================
# 1️⃣ Install Libraries
# =========================================
!pip install -q transformers datasets scikit-learn

# =========================================
# 2️⃣ Imports
# =========================================
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    logging
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

logging.set_verbosity_error()

# =========================================
# 3️⃣ Load RE Dataset
# =========================================
with open("/content/CN-dataset_relations_fixed.json") as f:
    data = json.load(f)

texts = [d["input"] for d in data]
labels = [d["output"].strip() for d in data]

label_list = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

labels = [l if l in label2id else "no_relation" for l in labels]

# =========================================
# 4️⃣ Train / Val / Test Split
# =========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

def make_dataset(texts, labels):
    return Dataset.from_dict({
        "text": texts,
        "label": [label2id[l] for l in labels]
    })

train_dataset = make_dataset(train_texts, train_labels)
val_dataset = make_dataset(val_texts, val_labels)
test_dataset = make_dataset(test_texts, test_labels)

# =========================================
# 5️⃣ Tokenization
# =========================================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

for d in [train_dataset, val_dataset, test_dataset]:
    d.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =========================================
# 6️⃣ Model
# =========================================
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# =========================================
# 7️⃣ Metrics
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# =========================================
# 8️⃣ Training Args
# =========================================
training_args = TrainingArguments(
    output_dir="./roberta-relation",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# =========================================
# 9️⃣ Trainer
# =========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =========================================
# 🔟 Train
# =========================================
trainer.train()

# =========================================
# 🧪 TEST SET EVALUATION
# =========================================
test_metrics = trainer.evaluate(eval_dataset=test_dataset)

print("\n📊 TEST SET RESULTS:")
print(f"Accuracy : {test_metrics['eval_accuracy']:.4f}")
print(f"Precision: {test_metrics['eval_precision']:.4f}")
print(f"Recall   : {test_metrics['eval_recall']:.4f}")
print(f"F1 Score : {test_metrics['eval_f1']:.4f}")

# =========================================
# 1️⃣1️⃣ Load NER Output
# =========================================
with open("/content/ensemble_entities.json") as f:
    test_data = json.load(f)

# =========================================
# 1️⃣2️⃣ Clean Entities
# =========================================
def clean_entities(entities):
    cleaned = []
    for e in entities:
        text = e["entity"].strip().lower()

        if len(text) < 4:
            continue
        if text in ["the", "this", "that", "data"]:
            continue
        if e["label"] == "OTHER":
            continue

        cleaned.append(e)

    return cleaned[:5]   # 🔥 LIMIT TO TOP 5

# =========================================
# 1️⃣3️⃣ Direction-Aware Prediction
# =========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def predict_best_relation(e1, t1, e2, t2, text):
    def run(inp):
        inputs = tokenizer(inp, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model(**inputs)
            probs = torch.softmax(out.logits, dim=1)
            conf, pred = torch.max(probs, dim=1)
        return id2label[pred.item()], conf.item()

    input1 = f"Sentence: {text} Entity1: {e1} ({t1}). Entity2: {e2} ({t2})."
    input2 = f"Sentence: {text} Entity1: {e2} ({t2}). Entity2: {e1} ({t1})."

    r1, c1 = run(input1)
    r2, c2 = run(input2)

    if c1 >= c2:
        return e1, t1, r1, e2, t2, c1
    else:
        return e2, t2, r2, e1, t1, c2

# =========================================
# 1️⃣4️⃣ Predict Relations
# =========================================
triplets = []
seen = set()

for item in test_data:
    text = item["text"]
    entities = clean_entities(item.get("entities", []))

    for i in range(len(entities)):
        for j in range(i+1, len(entities)):

            e1 = entities[i]["entity"]
            t1 = entities[i]["label"]
            e2 = entities[j]["entity"]
            t2 = entities[j]["label"]

            if e1.lower() == e2.lower():
                continue

            e1, t1, rel, e2, t2, conf = predict_best_relation(e1, t1, e2, t2, text)

            # 🔥 Strong filtering
            if rel == "no_relation" or conf < 0.8:
                continue

            key = (e1.lower(), rel, e2.lower())
            if key in seen:
                continue
            seen.add(key)

            triplets.append({
                "text": text,
                "entity1": e1,
                "entity1_type": t1,
                "relation": rel,
                "entity2": e2,
                "entity2_type": t2,
                "confidence": round(conf, 3)
            })

# =========================================
# 1️⃣5️⃣ Save Output
# =========================================
with open("final_triplets_level2.json", "w") as f:
    json.dump(triplets, f, indent=4)

print("\n✅ FINAL LEVEL-2 triplets saved to final_triplets_level2.json")

Map:   0%|          | 0/1756 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'eval_loss': '0.9263', 'eval_accuracy': '0.6837', 'eval_precision': '0.6133', 'eval_recall': '0.6837', 'eval_f1': '0.6456', 'eval_runtime': '1.312', 'eval_samples_per_second': '74.67', 'eval_steps_per_second': '9.905', 'epoch': '1'}


...

{'eval_loss': '0.7162', 'eval_accuracy': '0.8571', 'eval_precision': '0.8595', 'eval_recall': '0.8571', 'eval_f1': '0.8513', 'eval_runtime': '1.297', 'eval_samples_per_second': '75.54', 'eval_steps_per_second': '10.02', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.4658', 'grad_norm': '34.85', 'learning_rate': '1.836e-06', 'epoch': '4.545'}
{'eval_loss': '0.6796', 'eval_accuracy': '0.8571', 'eval_precision': '0.8588', 'eval_recall': '0.8571', 'eval_f1': '0.854', 'eval_runtime': '1.295', 'eval_samples_per_second': '75.66', 'eval_steps_per_second': '10.04', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '469.6', 'train_samples_per_second': '18.7', 'train_steps_per_second': '2.342', 'train_loss': '0.647', 'epoch': '5'}
{'eval_loss': '0.7427', 'eval_accuracy': '0.7959', 'eval_precision': '0.7974', 'eval_recall': '0.7959', 'eval_f1': '0.7952', 'eval_runtime': '1.3', 'eval_samples_per_second': '75.4', 'eval_steps_per_second': '10', 'epoch': '5'}

📊 TEST SET RESULTS:
Accuracy : 0.7959
Precision: 0.7974
Recall   : 0.7959
F1 Score : 0.7952

✅ FINAL LEVEL-2 triplets saved to final_triplets_level2.json
